In [1]:
import numpy as np
import pandas as pd
import tensorflow_hub as hub
import tensorflow as tf
# import tensorflow_text
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

In [2]:
df = pd.read_csv('data/weighted_score_above_08.csv')

C:\Users\Le Tam Quang\AppData\Local\Temp\ipykernel_15432\2037137802.py:1: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/weighted_score_above_08.csv')


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 498094 entries, 0 to 498093
Data columns (total 24 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   recommendationid                498094 non-null  int64  
 1   appid                           498094 non-null  int64  
 2   game                            498061 non-null  object 
 3   author_steamid                  498094 non-null  int64  
 4   author_num_games_owned          498094 non-null  int64  
 5   author_num_reviews              498094 non-null  int64  
 6   author_playtime_forever         498094 non-null  int64  
 7   author_playtime_last_two_weeks  498094 non-null  int64  
 8   author_playtime_at_review       498094 non-null  int64  
 9   author_last_played              498094 non-null  int64  
 10  language                        498094 non-null  object 
 11  review                          498094 non-null  object 
 12  timestamp_create

In [4]:
df_pos = df.loc[df['voted_up'] == 1][:30000]
df_neg = df.loc[df['voted_up'] == 0][:30000]
df = pd.concat([df_pos, df_neg], ignore_index=True)

In [5]:
def clean(texts):
    return texts.strip()

In [6]:
''' 
SBERT can capture semantic relationships and support multilingual data,
it has high quality embedding but the embedding process may took time. 
The feature space of SBERT is 384 and has dense results.
'''

' \nSBERT can capture semantic relationships and support multilingual data,\nit has high quality embedding but the embedding process may took time. \nThe feature space of SBERT is 384 and has dense results.\n'

In [7]:
texts = df['review'].astype(str).apply(clean).tolist()
sbert_mul = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device='cuda')

batch_size, l = 512, len(texts)
embedded_sbert_mul = []
for i in tqdm(range(0, l, batch_size), desc='Embedding using multilingual SBERT'):
    batch = texts[i:min(l, i+batch_size)]
    emb_sbert_mul = sbert_mul.encode(
        batch,
        show_progress_bar=False,
        convert_to_numpy=True
    )
    embedded_sbert_mul.append(emb_sbert_mul)

Embedding using multilingual SBERT: 100%|████████████████████████████████████████████| 118/118 [02:57<00:00,  1.50s/it]


In [8]:
embedded_sbert_mul = pd.DataFrame(np.vstack(embedded_sbert_mul), columns=[f'emb{i}' for i in range(384)])
df_sbert_mul = pd.concat([df.iloc[:, :12], embedded_sbert_mul, df.iloc[:, 12:]], axis=1)

In [9]:
df_sbert_mul.to_csv('results/df_sbert_mul_balanced.csv', index=False)

In [ ]:
'''
Universal Sentence Encoder can capture semantic relationships and also support multilingual datasets like SBERT.
It has lower quality embeddings but faster speed than SBERT.
The feature space of it is 512.
'''

In [ ]:
texts = df['review'].astype(str).apply(clean).tolist()
# Requires tensorflow_text but failed to pip install it in Python 3.11.5
use_mul = hub.load('https://tfhub.dev/google/universal-sentence-encoder-multilingual/3')

batch_size, l = 512, len(texts)
embedded_use_mul = []
for i in tqdm(range(0, l, batch_size), desc='Embedding using multilingual USE'):
    batch = texts[i:min(l, i+batch_size)]
    emb_use_mul = use_mul(batch)
    embedded_use_mul.append(emb_use_mul)

In [ ]:
embedded_use_mul = pd.DataFrame(np.vstack(embedded_use_mul), columns=[f'emb{i}' for i in range(512)])
df_use_mul = pd.concat([df.iloc[:, :12], embedded_use_mul, df.iloc[:, 12:]], axis=1)

In [ ]:
df_use_mul.to_csv('results/df_use_mul_balanced.csv', index=False)